In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/pre_cell_26.pickle

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['sample_discourse_id']
me:  2
trying: ['train_first_last']
me:  11
trying: ['get_n_grams_2']
me:  25
trying: ['warnings']
me:  0
trying: ['other_words_to_take_out']
me:  22
trying: ['counts_dict']
me:  26
trying: ['cols_to_merge']
me:  13
trying: ['test_names']
me:  26
trying: ['f']
me:  26
trying: ['train']
me:  22
trying: ['counter']
me:  11
trying: ['cols_to_display']
me:  14
trying: ['last_ones']
me:  14
trying: ['stop_english']
me:  22
trying: ['text']
me:  22
trying: ['data']
me:  12
trying: ['bigrams']
me:  23
trying: ['test_txt']
me:  1
trying: ['ax2']
me:  8
trying: ['train_last']
me:  11
trying: ['word_dict']
me:  12
trying: ['get_n_grams']
me:  23
trying: ['len_dict']
me:  12
trying: ['train_texts']
me:  26
trying: ['style']
me:  0
trying: ['tqdm']
me:  0
trying: ['av_per_essay']
me:  8
trying: ['all_gaps']
me:  20
trying: ['glob']
me:  0
trying: ['add_gap_rows']
me:  20
trying: ['stopword

In [4]:
%%RecordEvent
%%time
### cell 26 ###

### cell 26

# 0. Limit to first 5 essays
train_text_df = train_text_df[:5]

# 1. Tokenize each essay into one row per token, with token indices
tokenized = (
    train_text_df[['id', 'text']]
    .assign(tokens=train_text_df['text'].str.split(' '))
    .explode('tokens')
    .reset_index(drop=True)
)
# compute token_index per essay
tokenized['token_index'] = tokenized.groupby('id').cumcount()
print(1)

# 2. Expand the discourse annotations into one row per token index with B-/I- labels
labels = (
    train[['id', 'discourse_type', 'predictionstring']]
    .reset_index()
    .rename(columns={'index': 'disc_row'})
    .assign(
        token_index=lambda df: df['predictionstring']
            .str.split(' ')
            .apply(lambda lst: list(map(int, lst)))
    )
    .explode('token_index')
)
print(2)
labels['pos_in_discourse'] = labels.groupby('disc_row').cumcount()
labels['label'] = (
    ('B-' + labels['discourse_type'])
    .where(labels['pos_in_discourse'] == 0,
           'I-' + labels['discourse_type'])
)
labels = labels[['id', 'token_index', 'label']]
print(3)

# 3. Merge tokenized text with labels, defaulting missing labels to "O"
tokenized = (
    tokenized
    .merge(labels, on=['id', 'token_index'], how='left')
)
tokenized['label'] = tokenized['label'].fillna('O')
print(4)

# 4. Aggregate back to one list of labels per essay
entities_df = (
    tokenized
    .groupby('id')['label']
    .agg(list)
    .reset_index()
    .rename(columns={'label': 'entities'})
)
print(5)

# 5. Attach the entities column to the original DataFrame
train_text_df = train_text_df.merge(entities_df, on='id', how='left')
print(6)

1
2
3
4
5
6
CPU times: user 113 ms, sys: 48.3 ms, total: 162 ms
Wall time: 167 ms


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_26/checkpoints/post_cell_26_try_4.pickle

migration speed (bps): 737392132.2630244
---------------------------
variables to migrate:
train_texts 126104
IREWR_plug_2 61
counts_dict 696
train_text_df 48
print_colored_essay 144
entities_df 48
add_gap_rows_2 144
tokenized 48
glob 144
labels 48
sample_submission 48
IREWR_tmp 48
test_txt 120
last_ones 48
ax1 48
av_per_essay 48
sample_discourse_id 32
ax2 48
get_n_grams_2 144
trigrams 48
IREWR_tmp2 48
plot_ngram_2 144
df1 48
np 72
get_n_grams 144
IREWR_plug_1 61
FuncFormatter 1072
CountVectorizer 1072
tqdm 1072
cols_to_display 120
train_txt 126104
myword 28
train 48
total_gaps 48
txt_file 208
all_gaps 48
df 48
mylen 28
add_gap_rows 144
data 2813
t 166
bigrams 48
train_first_last 48
keys 120
myid 61
counter 28
stopwords 48
word_dict 589920
ax 48
plot_ngram 144
dt 57
train_first 48
len_dict 589920
other_words_to_take_out 152
i_1 28
stop_english 1688
train_last 48
plt 72
text 408
warnings 72
style 72
i_3 28
cols_to_merge 120
test_names 126104
f 65
pd 72
---------------------------
variab

In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['train', 'train_txt', 'sample_submission', 'stopwords']
Intermediate variables ['test_txt']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['sample_discourse_id', 'train']
Intermediate variables ['sample_submission']
Future variables ['train', 'train_txt']
Modified dataframes
  train
    Input columns: set()
    Changed columns: {'discourse_text', 'discourse_start', 'discourse_type_num', 'discourse_id', 'discourse_end', 'id', 'predictionstring', 'discourse_type'}
    Created columns: set()
    Deleted columns: set()
  sample_submission
    Input columns: set()
    Changed columns: {'id', 'predictionstring', 'class'}
    Created columns: set()
    Deleted columns: set()
======= Cell 2 =======
Input variables []
Active variables ['cols_to_display', 'train']
Intermediate variables []
Future variables ['sample_discourse_id', 'train_txt', 'train']
Modified dataframes
  train
    I

In [7]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_26_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[26], f)


In [8]:
opt_output = Out.get(4)

In [9]:
counts_dict_opt = counts_dict
train_text_df_opt = train_text_df
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/post_cell_26.pickle
assert counts_dict_opt == counts_dict
assert compare_df(train_text_df_opt, train_text_df)

import numpy as np
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['plt']
me:  0
trying: ['warnings']
me:  0
trying: ['entities_df']
me:  27
trying: ['pd']
me:  0
trying: ['style']
me:  0
trying: ['CountVectorizer']
me:  0
trying: ['text']
me:  22
trying: ['cols_to_merge']
me:  13
trying: ['test_names']
me:  26
trying: ['f']
me:  26
trying: ['train_texts']
me:  26
trying: ['IREWR_plug_2']
me:  16
trying: ['print_colored_essay']
me:  21
trying: ['counts_dict']
me:  28
trying: ['glob']
me:  0
trying: ['add_gap_rows_2']
me:  21
trying: ['sample_submission']
me:  2
trying: ['IREWR_tmp']
me:  16
trying: ['tokenized']
me:  27
trying: ['test_txt']
me:  1
trying: ['labels']
me:  27
trying: ['FuncFormatter']
me:  0
trying: ['sample_discourse_id']
me:  2
trying: ['last_ones']
me:  14
trying: ['ax1']
me:  8
trying: ['av_per_essay']
me:  8
trying: ['ax2']
me:  8
trying: ['get_n_grams_2']
me:  25
trying: ['trigrams']
me:  25
trying: ['IREWR_tmp2']
me:  17
trying: ['orig_output']

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().